# Evaluation Notebook — AI Lawyer IPC Assistant

This notebook produces the four metrics reported in Section 9 of the project report:

1. **Section Citation Accuracy** — % of answers citing the correct IPC section
2. **Hallucinated Section Rate** — % of cited sections not in retrieved context
3. **Retrieval Hit Rate (Top-3)** — whether expected section appears in top-3 results
4. **End-to-End Latency** — time from query to response (seconds)

**Prerequisites:**
- `GEMINI_API_KEY` set in `.env` or as an environment variable
- ChromaDB index built: `python scripts/build_index.py`
- Dependencies installed: `pip install -r requirements.txt`

## Setup & Imports

In [ ]:
import os
import re
import json
import time
from dotenv import load_dotenv

load_dotenv()

print('GEMINI_API_KEY set:', bool(os.getenv('GEMINI_API_KEY')))

In [ ]:
from app.services.retriever import retrieve_ipc_sections
from app.services.rag import answer_ipc_query

print('Retriever and pipeline imported successfully.')

## Helper Functions

In [ ]:
def extract_section_numbers(text):
    """Extract IPC section numbers from text using regex.
    Matches: 'Section 379', 'Section 366A', 'S. 302', '§ 420'
    """
    patterns = [
        r'[Ss]ection\s+(\d+[A-Z]?)',
        r'[Ss]\.\s*(\d+[A-Z]?)',
        r'§\s*(\d+[A-Z]?)',
    ]
    numbers = []
    for pat in patterns:
        for m in re.findall(pat, text):
            # Convert to int if purely numeric, keep string if alphanumeric
            try:
                numbers.append(int(m))
            except ValueError:
                numbers.append(m)
    return numbers

print('Helper functions defined.')

## Test Queries

Gold-standard queries with known expected IPC sections, derived from the
project's documented verification queries.

In [ ]:
TEST_QUERIES = [
    {
        'query': 'What is the punishment for theft?',
        'expected_sections': [378, 379],
        'description': 'Theft — defined in S378, punishment in S379'
    },
    {
        'query': 'What is the punishment for murder?',
        'expected_sections': [302],
        'description': 'Murder — punishment in S302'
    },
    {
        'query': 'What is the punishment for cheating?',
        'expected_sections': [417, 418, 419],
        'description': 'Cheating — defined in S415, punishment in S417/S418/S419'
    },
    {
        'query': 'Define criminal breach of trust.',
        'expected_sections': [405, 406],
        'description': 'Criminal breach of trust — defined in S405, punishment in S406'
    },
]

print(f'{len(TEST_QUERIES)} test queries loaded.')

## Metric 1: Retrieval Hit Rate (Top-3)

Measures whether the correct IPC section appears in the top-3 retrieved results
for each query. This isolates retrieval quality from generation quality.

In [ ]:
retrieval_results = []

for tq in TEST_QUERIES:
    retrieved = retrieve_ipc_sections(tq['query'], k=3)
    retrieved_sections = [r['section'] for r in retrieved]
    
    # Check if ANY expected section is in the retrieved top-3
    hit = any(exp in retrieved_sections for exp in tq['expected_sections'])
    
    retrieval_results.append({
        'query': tq['query'],
        'retrieved_sections': retrieved_sections,
        'expected_sections': tq['expected_sections'],
        'hit': hit
    })
    
    print(f"Query: {tq['query']}")
    print(f"  Expected: {tq['expected_sections']}")
    print(f"  Retrieved: {retrieved_sections}")
    print(f"  Hit: {'YES' if hit else 'NO'}")
    print()

hits = sum(r['hit'] for r in retrieval_results)
total = len(retrieval_results)
retrieval_hit_rate = hits / total * 100
print(f'=== Retrieval Hit Rate (Top-3): {hits}/{total} = {retrieval_hit_rate:.1f}% ===')

## Metrics 2, 3, 4: Full Pipeline Evaluation

Runs the complete RAG pipeline (retrieve → generate) for each query and
measures:
- **Section Citation Accuracy**: does the answer cite a section from the
  retrieved context?
- **Hallucinated Section Rate**: does the answer cite sections NOT in the
  retrieved context?
- **End-to-End Latency**: time from query to response

In [ ]:
all_results = []

for tq in TEST_QUERIES:
    print(f"\n--- Query: {tq['query']} ---")
    
    # Measure end-to-end latency
    start_time = time.time()
    try:
        answer, sections = answer_ipc_query(tq['query'])
        latency = time.time() - start_time
        error = None
    except Exception as exc:
        answer = ''
        sections = []
        latency = time.time() - start_time
        error = str(exc)
    
    # Sections retrieved (int or str)
    retrieved_section_nums = [s['section'] for s in sections]
    
    # Sections cited in the answer
    cited_sections = extract_section_numbers(answer)
    
    # Citation accuracy: any cited section is in the retrieved context
    correct_citations = 0
    hallucinated = 0
    total_citations = len(cited_sections)
    
    for cited in cited_sections:
        if cited in retrieved_section_nums:
            correct_citations += 1
        else:
            hallucinated += 1
    
    citation_accuracy = (correct_citations / total_citations * 100) if total_citations > 0 else 0
    hallucination_rate = (hallucinated / total_citations * 100) if total_citations > 0 else 0
    
    result = {
        'query': tq['query'],
        'answer_preview': answer[:200] + '...' if len(answer) > 200 else answer,
        'retrieved_sections': retrieved_section_nums,
        'cited_sections': cited_sections,
        'correct_citations': correct_citations,
        'hallucinated_citations': hallucinated,
        'total_citations': total_citations,
        'citation_accuracy': citation_accuracy,
        'hallucination_rate': hallucination_rate,
        'latency_seconds': round(latency, 2),
        'error': error
    }
    all_results.append(result)
    
    print(f"  Answer: {result['answer_preview']}")
    print(f"  Retrieved sections: {retrieved_section_nums}")
    print(f"  Cited sections: {cited_sections}")
    print(f"  Correct citations: {correct_citations}/{total_citations}")
    print(f"  Hallucinated: {hallucinated}")
    print(f"  Citation accuracy: {citation_accuracy:.1f}%")
    print(f"  Hallucination rate: {hallucination_rate:.1f}%")
    print(f"  Latency: {latency:.2f}s")
    if error:
        print(f"  ERROR: {error}")

## Summary Metrics Table

In [ ]:
print('=' * 80)
print('EVALUATION SUMMARY — AI Lawyer IPC Assistant')
print('=' * 80)
print()

# Metric 1: Retrieval Hit Rate
print(f'Metric 1 — Retrieval Hit Rate (Top-3): {retrieval_hit_rate:.1f}% ({hits}/{total})')
print()

# Metric 2: Section Citation Accuracy
valid_answers = [r for r in all_results if not r['error']]
if valid_answers:
    total_correct = sum(r['correct_citations'] for r in valid_answers)
    total_cited = sum(r['total_citations'] for r in valid_answers)
    overall_citation_accuracy = (total_correct / total_cited * 100) if total_cited > 0 else 0
    print(f'Metric 2 — Section Citation Accuracy: {overall_citation_accuracy:.1f}% ({total_correct}/{total_cited})')
else:
    print('Metric 2 — Section Citation Accuracy: N/A (all queries failed)')
print()

# Metric 3: Hallucinated Section Rate
if valid_answers:
    total_hallucinated = sum(r['hallucinated_citations'] for r in valid_answers)
    overall_hallucination_rate = (total_hallucinated / total_cited * 100) if total_cited > 0 else 0
    print(f'Metric 3 — Hallucinated Section Rate: {overall_hallucination_rate:.1f}% ({total_hallucinated}/{total_cited})')
else:
    print('Metric 3 — Hallucinated Section Rate: N/A (all queries failed)')
print()

# Metric 4: End-to-End Latency
if valid_answers:
    latencies = [r['latency_seconds'] for r in valid_answers]
    avg_latency = sum(latencies) / len(latencies)
    print(f'Metric 4 — End-to-End Latency: avg {avg_latency:.2f}s | min {min(latencies):.2f}s | max {max(latencies):.2f}s')
else:
    print('Metric 4 — End-to-End Latency: N/A (all queries failed)')
print()

print('Note: If any queries show errors, the Gemini free-tier API quota may')
print('be exhausted (20 requests/day for gemini-2.5-flash-lite). Wait 24h')
print('and re-run this notebook to get full results.')

## Detailed Results (JSON)

In [ ]:
print(json.dumps({
    'retrieval_hit_rate': retrieval_hit_rate,
    'retrieval_results': retrieval_results,
    'pipeline_results': all_results
}, indent=2, default=str))

## How to Re-Run

1. Ensure `GEMINI_API_KEY` is set in `.env`
2. Build the ChromaDB index if not already built: `python scripts/build_index.py`
3. Run all cells top to bottom
4. If you get 429 quota errors, wait 24 hours for the free-tier quota to reset